# project_11_fibril_binder — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [1]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

Python : 3.11.15
Platform: Linux-6.18.5-x86_64-with-glibc2.39
GPU    : NONE FOUND
Structure prediction on CPU is impractically slow.


## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [2]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

Note: you may need to restart the kernel to use updated packages.
Core install done.


In [3]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

# Environment stamp 2026-06-24T03:15:48 UTC
Bio            1.84
py3Dmol        2.4.0


numpy          2.4.6


pandas         3.0.3
matplotlib     3.11.0


seaborn        0.13.2
tqdm           4.68.3
requests       2.33.1


## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [4]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

Helpers ready: install_colabfold(), install_esmfold().


## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [5]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

seeds set to 0
logged: Ran 00_setup; environment stamped.


## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [6]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

Uncomment to mount Drive and set your working directory.


---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — fibril vs monomer, the epitope, conformational metrics

**Standard slot:** *define & explore.* **For Project 11 this means:** understand amyloid/fibril
structural biology, **choose the target conformation** (the ordered cross-β fibril surface, NOT the
disordered monomer), pick the fibril-surface **epitope residues**, write down the binder + the
**conformational-specificity** metrics, and run a deterministic **mock** mini-run as your
"hello-world" (D0).

Run `00_setup.ipynb` first in this session. A real binder campaign wants an **A100** (see
`MANUAL.md §2`); everything here runs on a no-GPU **mock** backend so you can build the plumbing
anywhere, then switch to the real backend on Colab Pro / A100.

> **The whole project in one sentence:** design a binder that grips the **fibril** conformation of tau
> (Alzheimer's) or α-synuclein (Parkinson's) and **ignores the monomer** — the basis of a
> conformation-selective diagnostic (PET tracer / assay) or an aggregation modulator.

## Why conformational specificity is the hard part

Tau and α-synuclein are **intrinsically disordered as monomers** — no single stable fold. In disease
they stack into **amyloid fibrils**: an ordered, repetitive **cross-β** core with a defined,
solvent-exposed surface (solved by cryo-EM). A useful binder here must do something a normal binder
does not have to do: **discriminate two conformations of the same protein.** It must recognize the
fibril surface and **reject the monomer**, because the monomer is abundant everywhere and a
cross-reactive binder is useless as a fibril-specific tracer.

This is genuinely hard, and **most designs will NOT be selective.** Be honest about it: report the
fraction that pass the fibril filter AND clear the monomer counter-test, not just the best one.

## The metrics, precisely (binder metrics + the specificity gap)

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the binder | thermostability / ΔG |
| **pae_interaction** | Å | AF2-Multimer error across the **binder–fibril interface** (key binder metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| rosetta_dG | REU | interface energy (more negative = stronger) | a guarantee it binds |
| shape complementarity | 0–1 | interface packing quality | epitope correctness |
| **pae_monomer** | Å | pae_interaction vs the **monomer** conformer (counter-test) | a measured off-rate |
| **specificity_gap** | Å | `pae_monomer − pae_fibril` (positive & large = prefers fibril) | a measured fold-selectivity |

The shared `"binder"` cutoffs: **scRMSD ≤ 2.5, pLDDT ≥ 80, pae_interaction ≤ 10, rosetta_dG ≤ −30,
sc ≥ 0.6.** `pae_interaction` is the single most important binder metric — but a low value is
*confidence*, **not** affinity. The **`specificity_gap` is the project's signature metric** (notebook
04): it is a teaching proxy on a model metric, NOT a measured selectivity, and the monomer is
disordered so its model carries extra uncertainty. A passing, "selective" design is a **hypothesis**
until a **fibril-vs-monomer ELISA/SPR** (notebook 05).

## Setup paths

In [7]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_11_fibril_binder/notebooks


## 1 · Target conformation + fibril-surface epitope

The design target is **one protofilament surface** of an amyloid fibril — tau paired-helical filament
(PHF) or an α-synuclein fibril — and the hotspots are the **exposed cross-β surface residues** the
binder grips. Steering the binder onto a fibril-specific surface (one that is buried or simply absent
in the disordered monomer) is what makes it *conformation-selective*. Fetch the candidate cryo-EM
fibrils with `data/download_data.py` (tau PHF **5O3L / 5O3T**, α-syn fibril **6CU7 / 6H6B** —
**verify on RCSB**), isolate a protofilament, keep the ordered core, and read the exposed surface
residues off the structure.

Below we just *declare* an EXAMPLE epitope set so the notebook runs end-to-end; **replace it with the
residues you derive from the actual fibril surface** (numbering depends on the PDB you verify).

In [8]:
import binder_tools as bt

# Two candidate amyloid targets (pick one to design against; cross-amyloid specificity is an extension).
#   TAU_PHF      tau paired-helical filament   (candidate cryo-EM: 5O3L / 5O3T — VERIFY on RCSB)
#   ASYN_FIBRIL  alpha-synuclein fibril        (candidate cryo-EM: 6CU7 / 6H6B — VERIFY on RCSB)
TARGET = "TAU_PHF"                  # one protofilament surface (you extract this from 5O3L/5O3T)

# EXAMPLE fibril-surface hotspots — VERIFY/REPLACE from the cryo-EM fibril surface (data/README.md).
# Real numbering depends on the PDB you clean; tau PHF ordered core is ~ residues 306-378 (R3-R4 repeats).
HOTSPOTS = bt.parse_hotspots("A306,A310,A315,A320")   # EXAMPLE_DATA placeholder fibril-surface residues
print("target  :", TARGET, " (the FIBRIL conformation — not the disordered monomer)")
print("hotspots:", HOTSPOTS, " (EXAMPLE — replace with your verified fibril-surface residues)")
print("\nCounter-test conformer: the disordered MONOMER (an AFDB/ensemble model — a modeling caveat).")

target  : TAU_PHF  (the FIBRIL conformation — not the disordered monomer)
hotspots: ('A306', 'A310', 'A315', 'A320')  (EXAMPLE — replace with your verified fibril-surface residues)

Counter-test conformer: the disordered MONOMER (an AFDB/ensemble model — a modeling caveat).


## 2 · Mock hello-world: a tiny two-paradigm mini-run

`scripts/binder_tools.py` exposes both paradigms behind one API:
`generate_binders_bindcraft(...)` and `generate_binders_rfdiffusion(...)`, plus `af2_multimer(...)`
(the FIBRIL-state scorer) and the project's HARD-PART helpers `conformational_specificity(...)` /
`specificity_gap(...)`. The **mock** backend is deterministic and GPU-free so you can develop the
plumbing. **Never report mock numbers as real** — they are `SYNTHETIC` by construction.

In [9]:
# A few designs from each paradigm, scored by mock AF2-Multimer (FIBRIL state). Numbers are SYNTHETIC.
bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=3, tool="mock")
rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=3, tool="mock")
bt.score_designs(bc, tool="mock")
bt.score_designs(rf, tool="mock")

d = bc[0]
print("example BindCraft design:")
print("  id   :", d.design_id)
print("  len  :", d.length, "aa")
print("  seq  :", d.sequence)
print("  pae_interaction (fibril) =", d.pae_interaction, " scrmsd =", d.scrmsd,
      " sc =", d.shape_complementarity, " (SYNTHETIC)")
print("  synthetic flag  :", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.")

example BindCraft design:
  id   : EXAMPLE_DATA_bindcraft_0000
  len  : 75 aa
  seq  : WEKCRYFQWEKQHEAVRNAGRYKGHIKVWSPCDSTCHNAGHIPQHEFQMYTVMEPCMNACMEPGREKGRNAVRSA
  pae_interaction (fibril) = 6.0  scrmsd = 1.92  sc = 0.77  (SYNTHETIC)
  synthetic flag  : True -> SYNTHETIC — mock backend, not a real design/prediction

Reminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.


## 3 · The HARD PART, previewed: fibril vs monomer

A fibril binder is only useful if it **prefers the fibril over the monomer**. `conformational_specificity()`
scores the binder against each conformer; `specificity_gap()` = `pae_monomer − pae_fibril` (positive &
large ⇒ prefers the fibril). This is the heart of notebook 04. Mock numbers are SYNTHETIC and the
monomer/fibril bias here is a **teaching device**, not a claim of real selectivity.

In [10]:
bt.evaluate_specificity(bc, tool="mock")   # scores BOTH conformers, fills pae_fibril/pae_monomer/specificity_gap
bt.evaluate_specificity(rf, tool="mock")

for b in bc[:3]:
    sel = "fibril-selective?" if (b.specificity_gap or 0) > 0 else "NOT selective"
    print(f"{b.design_id}: pae_fibril={b.pae_fibril}  pae_monomer={b.pae_monomer}  "
          f"gap={b.specificity_gap}  -> {sel}  (SYNTHETIC)")
print("\nA positive gap means AF2 is MORE confident about the fibril complex than the monomer one.")
print("This is a HYPOTHESIS — only a fibril-vs-monomer ELISA/SPR (nb 05) proves selectivity.")

EXAMPLE_DATA_bindcraft_0000: pae_fibril=6.0  pae_monomer=18.0  gap=12.0  -> fibril-selective?  (SYNTHETIC)
EXAMPLE_DATA_bindcraft_0001: pae_fibril=11.0  pae_monomer=23.0  gap=12.0  -> fibril-selective?  (SYNTHETIC)
EXAMPLE_DATA_bindcraft_0002: pae_fibril=11.0  pae_monomer=20.0  gap=9.0  -> fibril-selective?  (SYNTHETIC)

A positive gap means AF2 is MORE confident about the fibril complex than the monomer one.
This is a HYPOTHESIS — only a fibril-vs-monomer ELISA/SPR (nb 05) proves selectivity.


## 4 · Epitope-coverage proxy (does it cover the fibril surface?)

A binder can only be a fibril tracer if it actually sits on the exposed fibril epitope.
`hotspot_overlap()` is a geometry proxy (fraction of fibril-surface hotspots contacted) — a teaching
stand-in for the structural epitope mapping you would confirm experimentally. Higher ⇒ more of the
fibril surface covered (but coverage alone does NOT imply monomer rejection — that is the gap test).

In [11]:
for b in bc[:3]:
    ov = bt.hotspot_overlap(b.contact_residues, HOTSPOTS)
    print(f"{b.design_id}: contacts {b.contact_residues} -> fibril-epitope coverage = {ov} (SYNTHETIC)")

EXAMPLE_DATA_bindcraft_0000: contacts ('A306', 'A310') -> fibril-epitope coverage = 0.5 (SYNTHETIC)
EXAMPLE_DATA_bindcraft_0001: contacts ('A306', 'A310', 'A315') -> fibril-epitope coverage = 0.75 (SYNTHETIC)
EXAMPLE_DATA_bindcraft_0002: contacts ('A306', 'A310') -> fibril-epitope coverage = 0.5 (SYNTHETIC)


## Visualize a binder–fibril complex (py3Dmol)

Use this to eyeball a predicted binder–fibril complex once you have a real PDB (from AF2-Multimer), or
just to inspect the cryo-EM fibril surface you are targeting.

In [12]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real AF2-Multimer prediction writes a complex PDB, or to view the fibril):
# show_complex("results/af2/top_fibril_complex.pdb")
# show_complex("data/inputs/5O3L_protofilament.pdb")
print("show_complex(pdb_path) ready.")

show_complex(pdb_path) ready.


## D0 checklist
- [ ] Fibril accessions verified on RCSB (tau **5O3L/5O3T**, α-syn **6CU7/6H6B** are candidates); protofilament + ordered core identified.
- [ ] **Target conformation chosen** (the fibril surface) + a fibril-surface **epitope list** (derived from the structure, not invented).
- [ ] A monomer model assembled for the counter-test (AFDB/ensemble — note the disorder caveat).
- [ ] One-paragraph definition of each metric **with** its "does not mean" note, including `specificity_gap`.
- [ ] Reproduced mock mini-run (both paradigms) with metrics + the monomer-vs-fibril gap printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria + controls (fibril-vs-monomer, scrambled-interface); `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the two-paradigm binder campaign to the fibril epitope.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — two-paradigm binder design vs the amyloid fibril

**Standard slot:** *design campaign.* **For Project 11 this is the core:** run **both** paradigms
against the **fibril-surface** epitope and assemble their pools (D2):
- **BindCraft** (one-shot hallucination, AF2-Multimer in the loop) — **50–200** designs.
- **RFdiffusion binder mode → ProteinMPNN** — **500–1000** backbones → sequences.

Then score every design with **AF2-Multimer** against the fibril (`pae_interaction` is the key binder
metric). The monomer counter-test comes in notebook 04 — here we generate against the fibril.

> **Compute honesty:** a real campaign at this scale wants an **A100** (Colab Pro+ or a cluster).
> Free **T4** = a *small fallback* (FreeBindCraft, small `num_designs`, a small RFdiffusion batch +
> ESMFold triage). The cells below run on the deterministic **mock** backend so the plumbing executes
> anywhere; the real calls + A100 notes are shown alongside. Run `00_setup.ipynb` first.

## Setup paths

In [13]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_11_fibril_binder/notebooks


## Version-verify the pinned upstreams (tools change!)

The binder tools live in fast-moving upstream repos. **Pin commits** and **verify the URLs still
exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin and log
it). The generation itself needs an A100; this check needs nothing.

In [14]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   BindCraft      https://github.com/martinpacesa/BindCraft        # e.g. pin <commit>
#   FreeBindCraft  https://github.com/cytokineking/FreeBindCraft     # free-tier fallback — VERIFY it exists; pin <commit>
#   RFdiffusion    https://github.com/RosettaCommons/RFdiffusion     # pin <commit>
#   ColabDesign    https://github.com/sokrypton/ColabDesign          # RFdiffusion-binder + ProteinMPNN; pin <commit>
#   ColabFold      https://github.com/sokrypton/ColabFold            # AF2-Multimer; pin <commit>
PINNED = {
    "BindCraft":     "https://github.com/martinpacesa/BindCraft",
    "FreeBindCraft": "https://github.com/cytokineking/FreeBindCraft",
    "RFdiffusion":   "https://github.com/RosettaCommons/RFdiffusion",
    "ColabDesign":   "https://github.com/sokrypton/ColabDesign",
    "ColabFold":     "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:14s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:14s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.")

  [200] BindCraft      https://github.com/martinpacesa/BindCraft


  [200] FreeBindCraft  https://github.com/cytokineking/FreeBindCraft


  [200] RFdiffusion    https://github.com/RosettaCommons/RFdiffusion


  [200] ColabDesign    https://github.com/sokrypton/ColabDesign


  [200] ColabFold      https://github.com/sokrypton/ColabFold

Non-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.
FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.


## 1 · Define the campaign

Same fibril target + fibril-surface hotspots as notebook 01. Set honest campaign sizes; the cells run
on `mock` so they execute anywhere. On Colab (A100) switch `TOOL_*` to the real backends — and
**shrink the numbers on a T4** (FreeBindCraft, a small RFdiffusion batch).

In [15]:
import binder_tools as bt
import pandas as pd

TARGET = "TAU_PHF"                  # the fibril conformation (tau PHF); ASYN_FIBRIL for the alpha-syn run
HOTSPOTS = bt.parse_hotspots("A306,A310,A315,A320")   # EXAMPLE — replace with your verified fibril-surface residues

# Honest campaign sizes (catalog): BindCraft 50-200, RFdiffusion 500-1000 backbones.
# We use small mock counts here so the dry run is fast; scale up with the real backend on A100.
N_BINDCRAFT   = 60      # -> 50-200 on A100; fewer (FreeBindCraft) on T4
N_RFDIFFUSION = 200     # -> 500-1000 backbones on A100; small batch on T4

TOOL_BINDCRAFT   = "mock"   # -> "bindcraft" / "freebindcraft" on Colab
TOOL_RFDIFFUSION = "mock"   # -> "rfdiffusion" on Colab
TOOL_AF2         = "mock"   # -> "af2" (ColabFold AF2-Multimer) on Colab

print(f"BindCraft   : n={N_BINDCRAFT}  tool={TOOL_BINDCRAFT}")
print(f"RFdiffusion : n={N_RFDIFFUSION} tool={TOOL_RFDIFFUSION}")
print(f"AF2-Multimer: tool={TOOL_AF2}")
print("target/hotspots:", TARGET, HOTSPOTS)

BindCraft   : n=60  tool=mock
RFdiffusion : n=200 tool=mock
AF2-Multimer: tool=mock
target/hotspots: TAU_PHF ('A306', 'A310', 'A315', 'A320')


## 2 · Paradigm #1 — BindCraft campaign (vs the fibril)

One-shot hallucination with AF2-Multimer in the loop. On A100 this produces 50–200 binders
pre-filtered on interface confidence; we still re-score with AF2-Multimer so the head-to-head with
RFdiffusion is apples-to-apples. The `mock` backend returns deterministic `SYNTHETIC` designs.

In [16]:
# Real call (Colab, A100): bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool="bindcraft")
#   free-tier fallback: tool="freebindcraft", smaller N. See MANUAL.md §2 / scripts/binder_tools.py TODOs.
#   The BindCraft target_pdb is the cleaned FIBRIL protofilament surface (NOT the monomer).
bindcraft = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool=TOOL_BINDCRAFT)
bt.score_designs(bindcraft, tool=TOOL_AF2)     # AF2-Multimer (fibril) -> pae_interaction, plddt, scrmsd, sc
print(f"BindCraft pool: {len(bindcraft)} designs (tool={TOOL_BINDCRAFT}; SYNTHETIC if mock)")
print("example:", bindcraft[0].design_id, "pae_interaction(fibril)=", bindcraft[0].pae_interaction)

BindCraft pool: 60 designs (tool=mock; SYNTHETIC if mock)
example: EXAMPLE_DATA_bindcraft_0000 pae_interaction(fibril)= 6.0


## 3 · Paradigm #2 — RFdiffusion binder campaign → ProteinMPNN

Diffuse binder backbones docked at the fibril-surface hotspots, then ProteinMPNN designs sequences,
then AF2-Multimer re-predicts each complex. On A100 this is 500–1000 backbones (the per-backbone hit
rate is low — that is normal, and even lower here because the flat cross-β surface is a hard target).
The `mock` backend stands in for the whole chain.

In [17]:
# Real call (Colab, A100): bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION,
#   tool="rfdiffusion", mpnn_temperature=0.1, num_seq_per_backbone=8). AF2-Multimer is the slow step.
rfdiff = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION, tool=TOOL_RFDIFFUSION)
bt.score_designs(rfdiff, tool=TOOL_AF2)
print(f"RFdiffusion pool: {len(rfdiff)} designs (tool={TOOL_RFDIFFUSION}; SYNTHETIC if mock)")
print("example:", rfdiff[0].design_id, "pae_interaction(fibril)=", rfdiff[0].pae_interaction)

RFdiffusion pool: 200 designs (tool=mock; SYNTHETIC if mock)
example: EXAMPLE_DATA_rfdiffusion_0000 pae_interaction(fibril)= 11.0


## 4 · Assemble + persist both pools

Write one tidy CSV per paradigm (plus a combined one). These feed notebook 03 (the shared filter) and
notebook 04 (the conformational-specificity test). We add an EXAMPLE physics column (`rosetta_dG`)
here so the binder physics layer has something to act on in the dry run — on Colab these come from
FreeBindCraft/PyRosetta; for `mock` they are SYNTHETIC.

In [18]:
import pandas as pd

def pool_to_df(designs):
    rows = []
    for d in designs:
        # In the mock dry run we attach an EXAMPLE_DATA interface energy so Layer 3 (physics) is
        # exercised. On Colab, replace with the real FreeBindCraft/PyRosetta rosetta_dG + solubility.
        rdg = -45.0 + (bt._hashints("dG", d.design_id) % 40)   # SYNTHETIC, range ~ -45..-6 REU
        rows.append(dict(
            design_id=d.design_id, paradigm=d.paradigm, target=d.target,
            length=d.length, sequence=d.sequence,
            plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
            shape_complementarity=d.shape_complementarity,
            rosetta_dG=round(float(rdg), 2), solubility=0.3,
            contact_residues=",".join(d.contact_residues),
            hotspot_overlap=bt.hotspot_overlap(d.contact_residues, d.hotspots),
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

df_bc = pool_to_df(bindcraft); df_bc.to_csv("results/bindcraft_designs.csv", index=False)
df_rf = pool_to_df(rfdiff);    df_rf.to_csv("results/rfdiffusion_designs.csv", index=False)
combined = pd.concat([df_bc, df_rf], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

print("wrote results/bindcraft_designs.csv   ", df_bc.shape)
print("wrote results/rfdiffusion_designs.csv ", df_rf.shape)
print("wrote results/all_designs.csv         ", combined.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.")
combined.head(4)

wrote results/bindcraft_designs.csv    (60, 14)
wrote results/rfdiffusion_designs.csv  (200, 14)
wrote results/all_designs.csv          (260, 14)

ALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.


,design_id,paradigm,target,length,sequence,plddt,pae_interaction,scrmsd,shape_complementarity,rosetta_dG,solubility,contact_residues,hotspot_overlap,synthetic
0,EXAMPLE_DATA_bindcraft_0000,bindcraft,TAU_PHF,75,WEKCRYFQWEKQHEAVRNAGRYKGHIKVWSPCDSTCHNAGHIPQHE...,72.0,6.0,1.92,0.77,-38.0,0.3,"A306,A310",0.50,True
1,EXAMPLE_DATA_bindcraft_0001,bindcraft,TAU_PHF,71,ITQMIFQWSKQMEPQHEFCWSKGHYTQRSFCHNALHIKGMSFGRYP...,91.0,11.0,2.01,0.81,-19.0,0.3,"A306,A310,A315",0.75,True
2,EXAMPLE_DATA_bindcraft_0002,bindcraft,TAU_PHF,86,WEACRSTLDSPGRSKQWETGHYACMSAVRSALDNACHYPVMNKVHI...,79.0,11.0,3.59,0.54,-35.0,0.3,"A306,A310",0.50,True
3,EXAMPLE_DATA_bindcraft_0003,bindcraft,TAU_PHF,82,QDNTVRNTVRYAQMIAQDNFCMSKGMYAVHETLRETGDSKCRSFCM...,72.0,16.0,1.22,0.77,-38.0,0.3,A306,0.25,True


## D2 checklist
- [ ] BindCraft pool generated at honest scale (50–200 on A100; FreeBindCraft/small on T4) against the **fibril** surface.
- [ ] RFdiffusion-binder pool generated (500–1000 backbones → ProteinMPNN on A100); low per-backbone hit rate expected (flat cross-β target).
- [ ] Every design scored by AF2-Multimer vs the fibril (`pae_interaction` parsed); both pools written to `results/`.
- [ ] Design log: every config + seed + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured; 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on both pools.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter (binder cutoffs)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 11** you build `fp.Design` **binder** objects from both pools, call
`fp.run_pipeline(..., design_type="binder")`, and `fp.report(...)` the survival funnel + ranked CSV,
**per paradigm** so the head-to-head is fair (D3 part 1).

> The filter here ranks designs on the **fibril** interface. The conformational-specificity counter-test
> (monomer vs fibril) is notebook 04 — the filter enriches for good fibril binders first; selectivity
> is applied on top.

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back. This notebook *imports* it.

Run `00`–`02` first so `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` exist.

## Setup paths

In [19]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_11_fibril_binder/notebooks


## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"binder"` cutoffs: scRMSD ≤ 2.5, pLDDT ≥ 80, pae ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6.

In [20]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])

Loaded shared filtering_pipeline from: /home/user/biofx_python/denovo_protein_design_course/shared/filtering_pipeline.py
binder cutoffs: {'scrmsd': 2.5, 'plddt': 80, 'pae': 10, 'rosetta_dG': -30, 'sc': 0.6}


## Build `Design` (binder) objects from the pools

Map each pool row onto `fp.Design` with `design_type="binder"`. The binder metrics drive the layers:
`scrmsd`/`plddt`/`pae_interaction` (Layer 1 self-consistency), and `rosetta_dG`/`shape_complementarity`/`solubility`
(Layer 3 physics). We keep `paradigm` + `hotspot_overlap` in `extra` for the head-to-head + epitope
analysis in notebook 04. (Mock has no independent orthogonal predictor, so we run Layers 1+3 here;
on Colab add a second predictor for Layer 2.)

In [21]:
import os
import pandas as pd

# Regenerate the pools if a fresh session lost them (deterministic mock).
if not (os.path.exists("results/bindcraft_designs.csv") and os.path.exists("results/rfdiffusion_designs.csv")):
    import binder_tools as bt
    TARGET, HOTSPOTS = "TAU_PHF", bt.parse_hotspots("A306,A310,A315,A320")
    bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=60, tool="mock");  bt.score_designs(bc, tool="mock")
    rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=200, tool="mock"); bt.score_designs(rf, tool="mock")
    def _q(designs, p):
        rows=[dict(design_id=d.design_id, paradigm=d.paradigm, length=d.length, sequence=d.sequence,
                   plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
                   shape_complementarity=d.shape_complementarity,
                   rosetta_dG=round(-45.0+(bt._hashints("dG",d.design_id)%40),2), solubility=0.3,
                   hotspot_overlap=bt.hotspot_overlap(d.contact_residues,d.hotspots), synthetic=d.synthetic)
              for d in designs]
        pd.DataFrame(rows).to_csv(p, index=False)
    _q(bc, "results/bindcraft_designs.csv"); _q(rf, "results/rfdiffusion_designs.csv")

df_bc = pd.read_csv("results/bindcraft_designs.csv")
df_rf = pd.read_csv("results/rfdiffusion_designs.csv")

def row_to_binder(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type="binder",
        plddt=r.get("plddt"), pae_interaction=r.get("pae_interaction"), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd predictor on Colab
        rosetta_dG=r.get("rosetta_dG"), shape_complementarity=r.get("shape_complementarity"),
        solubility=r.get("solubility", 0.3),
        extra={"paradigm": r.get("paradigm"), "hotspot_overlap": r.get("hotspot_overlap")},
    )

binders_bc = [row_to_binder(r) for _, r in df_bc.iterrows()]
binders_rf = [row_to_binder(r) for _, r in df_rf.iterrows()]
print(f"built {len(binders_bc)} BindCraft + {len(binders_rf)} RFdiffusion binder Designs")

built 60 BindCraft + 200 RFdiffusion binder Designs


## Run the pipeline — per paradigm (fair head-to-head)

`run_pipeline(design_type="binder")` applies the binder cutoffs in order and returns a ranked
DataFrame with survival counts in `df.attrs`. We run **each paradigm separately** so the
survival-at-each-layer funnels are comparable. We use Layers 1+3 here (mock has no independent
orthogonal source; add Layer 2 on Colab with a second predictor).

In [22]:
def run_one(designs, label):
    df = fp.run_pipeline(designs, design_type="binder", use_layers=(1, 3))
    df["paradigm"] = label
    surv = df.attrs["survival"]; n = df.attrs["n_total"]
    passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: {n} designs, survival {surv}, all-layers hit rate = {passed}/{n} ({100*passed/max(n,1):.1f}%)")
    return df

ranked_bc = run_one(binders_bc, "bindcraft")
ranked_rf = run_one(binders_rf, "rfdiffusion")

ranked = pd.concat([ranked_bc, ranked_rf], ignore_index=True).sort_values(
    ["layers_passed", "score"], ascending=False).reset_index(drop=True)
ranked.to_csv("results/all_ranked.csv", index=False)
print("\nwrote results/all_ranked.csv", ranked.shape)
ranked.head(10)[["design_id", "paradigm", "layers_passed", "score",
                 "scrmsd", "plddt", "pae_interaction", "rosetta_dG"]]

bindcraft   : 60 designs, survival {'L1': 8, 'L3': 0}, all-layers hit rate = 0/60 (0.0%)
rfdiffusion : 200 designs, survival {'L1': 23, 'L3': 6}, all-layers hit rate = 6/200 (3.0%)

wrote results/all_ranked.csv (260, 21)


,design_id,paradigm,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG
0,EXAMPLE_DATA_rfdiffusion_0024,rfdiffusion,3,4.3967,1.22,92.0,4.0,-39.0
1,EXAMPLE_DATA_rfdiffusion_0008,rfdiffusion,3,4.1600,1.19,99.0,9.0,-43.0
2,EXAMPLE_DATA_rfdiffusion_0150,rfdiffusion,3,4.0967,1.36,86.0,4.0,-34.0
3,EXAMPLE_DATA_rfdiffusion_0148,rfdiffusion,3,3.6467,1.61,91.0,7.0,-34.0
4,EXAMPLE_DATA_rfdiffusion_0193,rfdiffusion,3,3.2067,1.99,89.0,7.0,-32.0
5,EXAMPLE_DATA_rfdiffusion_0188,rfdiffusion,3,3.1067,2.23,93.0,7.0,-37.0
6,EXAMPLE_DATA_bindcraft_0059,bindcraft,1,4.3167,0.84,94.0,4.0,-40.0
7,EXAMPLE_DATA_bindcraft_0010,bindcraft,1,4.0067,0.91,91.0,7.0,-42.0
8,EXAMPLE_DATA_bindcraft_0048,bindcraft,1,3.9033,0.88,98.0,8.0,-36.0
9,EXAMPLE_DATA_rfdiffusion_0041,rfdiffusion,1,3.7367,1.02,92.0,4.0,-21.0


## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Here we report the **combined**
pool for one comparable figure; the per-paradigm runs above are the rigorous version. Read the bars as
a funnel: steep drops show which layer discriminates.

In [23]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

all_binders = binders_bc + binders_rf
df_all = fp.run_pipeline(all_binders, design_type="binder", use_layers=(1, 3))
top = fp.report(df_all, top_n=15, save_prefix="results/p11")
print("\nsaved results/p11_survival.png + results/p11_ranked.csv")
top

Total designs: 260
  L1 survivors: 31  (11.9%)
  L3 survivors: 6  (2.3%)



saved results/p11_survival.png + results/p11_ranked.csv


,design_id,design_type,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG,tm_to_pdb
0,EXAMPLE_DATA_rfdiffusion_0024,binder,3,4.3967,1.22,92.0,4.0,-39.0,None
1,EXAMPLE_DATA_rfdiffusion_0008,binder,3,4.1600,1.19,99.0,9.0,-43.0,None
2,EXAMPLE_DATA_rfdiffusion_0150,binder,3,4.0967,1.36,86.0,4.0,-34.0,None
3,EXAMPLE_DATA_rfdiffusion_0148,binder,3,3.6467,1.61,91.0,7.0,-34.0,None
4,EXAMPLE_DATA_rfdiffusion_0193,binder,3,3.2067,1.99,89.0,7.0,-32.0,None
5,EXAMPLE_DATA_rfdiffusion_0188,binder,3,3.1067,2.23,93.0,7.0,-37.0,None
6,EXAMPLE_DATA_bindcraft_0059,binder,1,4.3167,0.84,94.0,4.0,-40.0,None
7,EXAMPLE_DATA_bindcraft_0010,binder,1,4.0067,0.91,91.0,7.0,-42.0,None
8,EXAMPLE_DATA_bindcraft_0048,binder,1,3.9033,0.88,98.0,8.0,-36.0,None
9,EXAMPLE_DATA_rfdiffusion_0041,binder,1,3.7367,1.02,92.0,4.0,-21.0,None


## Honest hit-rate accounting (per paradigm)

Report `N passing all layers / N generated` for **each** paradigm — this is the number the head-to-head
in notebook 04 builds on. Remember: survival is *enrichment*, not *correctness*, and it does NOT yet
account for conformational selectivity (that is notebook 04). Mock numbers are SYNTHETIC.

In [24]:
for label, df in [("bindcraft", ranked_bc), ("rfdiffusion", ranked_rf)]:
    n = len(df); passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: layers_passed distribution {df['layers_passed'].value_counts().sort_index().to_dict()}")
    print(f"{'':12s}  all-layers survivors = {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")

bindcraft   : layers_passed distribution {0: 52, 1: 8}
              all-layers survivors = 0/60 (0.0%)  [SYNTHETIC if mock]
rfdiffusion : layers_passed distribution {0: 177, 1: 17, 3: 6}
              all-layers survivors = 6/200 (3.0%)  [SYNTHETIC if mock]


## D3 (part 1) checklist
- [ ] `results/all_ranked.csv` produced by the **shared** module (`design_type="binder"`), not a one-off script.
- [ ] Survival-at-each-layer reported **per paradigm** (funnel figure `results/p11_survival.png`).
- [ ] Honest hit-rate accounting (N pass / N generated) for BindCraft and RFdiffusion.
- [ ] Mapping assumptions (which fields → which `Design` attributes) written down.

**Next:** `04_validate.ipynb` — the conformational-specificity test (fibril vs monomer) + head-to-head.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — conformational specificity (fibril vs monomer) + head-to-head

**Standard slot:** *validate (in silico).* **For Project 11 this is the HARD PART:** the
**conformational-specificity test** — model each survivor against the **monomer** and the **fibril**
and require it to **prefer the fibril** (`specificity_gap > 0`) — plus the BindCraft-vs-RFdiffusion
head-to-head and the **cross-amyloid** specificity extension (tau vs α-syn), with publication-style
figures (D3 part 2).

Needs `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` + `results/all_ranked.csv`
(from notebooks 02–03).

> **Why this notebook matters most.** A binder that scores beautifully on the fibril but ALSO binds the
> abundant monomer is useless as a fibril-specific tracer. Selectivity — not raw fibril affinity — is
> what makes this a diagnostic. Expect **most** designs to fail the monomer counter-test; report that
> honestly.

## Setup paths

In [25]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_11_fibril_binder/notebooks


## 1 · The conformational-specificity test (fibril vs monomer)

For every design we score **both** conformers with `conformational_specificity()` and compute
`specificity_gap = pae_monomer − pae_fibril`. **Positive & large ⇒ prefers the fibril** (what we
want). We regenerate the pools deterministically (mock) so this notebook is self-contained, then run
the two-state evaluation.

**Caveat, stated up front:** the monomer is intrinsically disordered, so its model (and therefore the
gap) carries extra uncertainty. The gap is a teaching proxy on a model metric — **not** a measured
fold-selectivity. The wet-lab fibril-vs-monomer assay (notebook 05) is what actually proves it.

In [26]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import binder_tools as bt

TARGET, HOTSPOTS = "TAU_PHF", bt.parse_hotspots("A306,A310,A315,A320")

# Rebuild the pools deterministically (mock) and run the TWO-STATE specificity evaluation.
bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=60, tool="mock");  bt.score_designs(bc, tool="mock")
rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=200, tool="mock"); bt.score_designs(rf, tool="mock")
bt.evaluate_specificity(bc, tool="mock")     # fills pae_fibril / pae_monomer / specificity_gap
bt.evaluate_specificity(rf, tool="mock")

def spec_df(designs, label):
    return pd.DataFrame([dict(
        design_id=d.design_id, paradigm=label, length=d.length,
        pae_fibril=d.pae_fibril, pae_monomer=d.pae_monomer,
        specificity_gap=d.specificity_gap, sequence=d.sequence,
        hotspot_overlap=bt.hotspot_overlap(d.contact_residues, d.hotspots),
    ) for d in designs])

spec = pd.concat([spec_df(bc, "bindcraft"), spec_df(rf, "rfdiffusion")], ignore_index=True)
spec.to_csv("results/specificity.csv", index=False)
print("wrote results/specificity.csv", spec.shape, " (SYNTHETIC mock numbers)")
print(spec[["design_id", "paradigm", "pae_fibril", "pae_monomer", "specificity_gap"]].head(6).to_string(index=False))

wrote results/specificity.csv (260, 8)  (SYNTHETIC mock numbers)
                  design_id  paradigm  pae_fibril  pae_monomer  specificity_gap
EXAMPLE_DATA_bindcraft_0000 bindcraft         6.0         18.0             12.0
EXAMPLE_DATA_bindcraft_0001 bindcraft        11.0         23.0             12.0
EXAMPLE_DATA_bindcraft_0002 bindcraft        11.0         20.0              9.0
EXAMPLE_DATA_bindcraft_0003 bindcraft        16.0         14.0             -2.0
EXAMPLE_DATA_bindcraft_0004 bindcraft         7.0         26.0             19.0
EXAMPLE_DATA_bindcraft_0005 bindcraft         5.0         15.0             10.0


## 2 · Define "fibril-selective" and count it (per paradigm)

A design is **fibril-selective** if (a) it is a decent fibril binder (`pae_fibril ≤ 10`, the shared
binder cutoff) **and** (b) it clears a conformational-selectivity margin (`specificity_gap ≥ GAP_MIN`).
Choose `GAP_MIN` and **justify it** — there is no universal value; it trades selectivity stringency
against yield. Report how many designs survive BOTH conditions, per paradigm.

In [27]:
PAE_FIBRIL_MAX = 10.0   # shared binder cutoff (good fibril binder)
GAP_MIN        = 4.0    # conformational-selectivity margin (JUSTIFY; tune in your report)

spec["good_fibril_binder"] = spec["pae_fibril"] <= PAE_FIBRIL_MAX
spec["fibril_selective"]   = spec["good_fibril_binder"] & (spec["specificity_gap"] >= GAP_MIN)

print(f"Selectivity definition: pae_fibril <= {PAE_FIBRIL_MAX} AND specificity_gap >= {GAP_MIN}")
for p, g in spec.groupby("paradigm"):
    nb = int(g["good_fibril_binder"].sum()); ns = int(g["fibril_selective"].sum()); n = len(g)
    print(f"  {p:12s}: good fibril binders {nb}/{n}; ALSO fibril-selective {ns}/{n} "
          f"({100*ns/max(n,1):.1f}%)  [SYNTHETIC if mock]")
print("\nExpect the selective fraction to be SMALL — rejecting the monomer is the hard requirement.")

Selectivity definition: pae_fibril <= 10.0 AND specificity_gap >= 4.0
  bindcraft   : good fibril binders 27/60; ALSO fibril-selective 25/60 (41.7%)  [SYNTHETIC if mock]
  rfdiffusion : good fibril binders 94/200; ALSO fibril-selective 86/200 (43.0%)  [SYNTHETIC if mock]

Expect the selective fraction to be SMALL — rejecting the monomer is the hard requirement.


In [28]:
# Figure: specificity gap distribution + the selectivity quadrant (pae_fibril vs pae_monomer).
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for p, g in spec.groupby("paradigm"):
    ax[0].hist(g["specificity_gap"].dropna(), bins=15, alpha=0.5, label=p)
ax[0].axvline(GAP_MIN, color="k", ls="--", lw=1, label=f"GAP_MIN={GAP_MIN}")
ax[0].set_xlabel("specificity_gap = pae_monomer - pae_fibril (Å; >0 prefers fibril)")
ax[0].set_ylabel("designs"); ax[0].set_title("Conformational selectivity"); ax[0].legend()

# pae_fibril (x) vs pae_monomer (y): selective designs are LOW-x, HIGH-y (upper-left).
for p, g in spec.groupby("paradigm"):
    ax[1].scatter(g["pae_fibril"], g["pae_monomer"], s=14, alpha=0.6, label=p)
lim = [spec[["pae_fibril", "pae_monomer"]].min().min() - 1, spec[["pae_fibril", "pae_monomer"]].max().max() + 1]
ax[1].plot(lim, lim, "k--", lw=1, label="no selectivity (y=x)")
ax[1].axvline(PAE_FIBRIL_MAX, color="grey", ls=":", lw=1)
ax[1].set_xlabel("pae_fibril (Å, lower = better fibril binder)")
ax[1].set_ylabel("pae_monomer (Å, higher = rejects monomer)")
ax[1].set_title("Selectivity quadrant (want upper-left)"); ax[1].legend()
fig.suptitle("Conformational specificity — fibril vs monomer (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p11_specificity.png", dpi=150); plt.show()
print("saved results/p11_specificity.png")

saved results/p11_specificity.png


## 3 · Head-to-head: BindCraft vs RFdiffusion

Same fair comparison as the binder-family template: hit rate (here, the **fibril-selective** rate is
the meaningful one) and the `pae_interaction` / interface-energy distributions. Report the
*distribution*, not the single best. Mock numbers are SYNTHETIC.

In [29]:
ranked = pd.read_csv("results/all_ranked.csv")
# attach specificity to the ranked survivors by design_id
spec_idx = spec.set_index("design_id")
ranked["specificity_gap"] = ranked["design_id"].map(spec_idx["specificity_gap"])
ranked["pae_monomer"]     = ranked["design_id"].map(spec_idx["pae_monomer"])
ranked["fibril_selective"]= ranked["design_id"].map(spec_idx["fibril_selective"]).fillna(False)

summary = []
for p, g in ranked.groupby("paradigm"):
    n = len(g)
    passed = int((g["layers_passed"] >= 3).sum())
    sel = int(g["fibril_selective"].sum())
    summary.append(dict(paradigm=p, n=n, all_layers_survivors=passed,
                        hit_rate_pct=round(100*passed/max(n,1), 1),
                        fibril_selective=sel, selective_pct=round(100*sel/max(n,1), 1),
                        median_pae=round(float(g["pae_interaction"].median()), 2),
                        median_gap=round(float(g["specificity_gap"].median()), 2)))
summary = pd.DataFrame(summary)
print("head-to-head summary (SYNTHETIC if mock):")
print(summary.to_string(index=False))

head-to-head summary (SYNTHETIC if mock):
   paradigm   n  all_layers_survivors  hit_rate_pct  fibril_selective  selective_pct  median_pae  median_gap
  bindcraft  60                     0           0.0                25           41.7        11.0         7.5
rfdiffusion 200                     6           3.0                86           43.0        11.0         7.0


In [30]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for p, g in ranked.groupby("paradigm"):
    surv = g[g["layers_passed"] >= 3]
    ax[0].hist(g["pae_interaction"].dropna(), bins=15, alpha=0.5, label=p)
    ax[1].hist(surv["rosetta_dG"].dropna(), bins=15, alpha=0.5, label=p)
ax[0].set_xlabel("pae_interaction (Å, fibril; lower better)"); ax[0].set_ylabel("designs"); ax[0].set_title("AF2-Multimer pae_interaction"); ax[0].legend()
ax[1].set_xlabel("rosetta_dG (REU, more negative better)"); ax[1].set_title("Interface energy (survivors)"); ax[1].legend()
fig.suptitle("BindCraft vs RFdiffusion (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p11_headtohead.png", dpi=150); plt.show()
print("saved results/p11_headtohead.png")

saved results/p11_headtohead.png


## 4 · Cross-amyloid specificity (tau vs α-syn) `[extension]`

A tau-fibril tracer should **not** light up α-synuclein fibrils (and vice versa) — otherwise it cannot
tell Alzheimer's from Parkinson's pathology. The cross-amyloid test scores each tau-designed binder
against the **α-synuclein** fibril as an off-target conformer. On Colab, build the α-syn fibril target
(6CU7/6H6B), re-run AF2-Multimer, and compute a **cross-amyloid gap** the same way. Here we scaffold
it with the mock backend (different `target` ⇒ different deterministic numbers) so the analysis shape
is in place.

In [31]:
# Cross-amyloid counter-test: score the TAU-designed binders against the ALPHA-SYN fibril (off-target).
# On Colab: replace target="ASYN_FIBRIL" with the cleaned 6CU7/6H6B protofilament and run real AF2-Multimer.
def cross_amyloid_gap(designs, off_target="ASYN_FIBRIL", tool="mock"):
    rows = []
    for d in designs:
        on  = bt.af2_multimer(d.sequence, target=d.target,   tool=tool, hotspots=d.hotspots, conformer="fibril")
        off = bt.af2_multimer(d.sequence, target=off_target, tool=tool, hotspots=d.hotspots, conformer="fibril")
        rows.append(dict(design_id=d.design_id, paradigm=d.paradigm,
                         pae_on_target=on["pae_interaction"], pae_off_target=off["pae_interaction"],
                         cross_amyloid_gap=round(off["pae_interaction"] - on["pae_interaction"], 3)))
    return pd.DataFrame(rows)

cross = cross_amyloid_gap(bc + rf, off_target="ASYN_FIBRIL", tool="mock")
cross.to_csv("results/cross_amyloid.csv", index=False)
print("wrote results/cross_amyloid.csv", cross.shape, " (SYNTHETIC mock numbers)")
print("cross-amyloid gap = pae(off-target alpha-syn) - pae(on-target tau); >0 means tau-preferring.")
print(cross.groupby("paradigm")["cross_amyloid_gap"].median().round(2).to_dict(),
      " (median cross-amyloid gap per paradigm; SYNTHETIC)")

wrote results/cross_amyloid.csv (260, 5)  (SYNTHETIC mock numbers)
cross-amyloid gap = pae(off-target alpha-syn) - pae(on-target tau); >0 means tau-preferring.
{'bindcraft': -0.5, 'rfdiffusion': 0.0}  (median cross-amyloid gap per paradigm; SYNTHETIC)


## 5 · Select the top conformation-selective candidates per paradigm

The D★ deliverable wants a **binder set** that is fibril-selective. Rank the fibril-selective survivors
by the fibril composite score, tie-break on a larger `specificity_gap`, and keep the top per paradigm.
Save the shortlist for the validation plan (notebook 05). If very few are selective, **say so** — that
is the honest, expected outcome.

In [32]:
# Bring the epitope-coverage proxy onto the ranked survivors (it lives in the specificity table).
ranked["hotspot_overlap"] = ranked["design_id"].map(spec_idx["hotspot_overlap"])

top_per = []
for p, g in ranked.groupby("paradigm"):
    sel = g[(g["layers_passed"] >= 3) & (g["fibril_selective"] == True)]
    sel = sel.sort_values(["score", "specificity_gap"], ascending=False).head(20)
    top_per.append(sel)
top = pd.concat(top_per, ignore_index=True)
top.to_csv("results/top_candidates.csv", index=False)
print("wrote results/top_candidates.csv:", top.shape, "(top<=20 fibril-selective per paradigm)")
print("fibril-selective top-set per paradigm:", top.groupby("paradigm").size().to_dict())
if len(top) == 0:
    print("NOTE: zero fibril-selective survivors — a legitimate outcome. Loosen GAP_MIN OR report a 0% selective rate honestly.")
cols = [c for c in ["design_id", "paradigm", "score", "pae_interaction", "specificity_gap", "hotspot_overlap"] if c in top.columns]
top.head(8)[cols] if len(top) else "no fibril-selective candidates at this GAP_MIN"

wrote results/top_candidates.csv: (5, 25) (top<=20 fibril-selective per paradigm)
fibril-selective top-set per paradigm: {'rfdiffusion': 5}


,design_id,paradigm,score,pae_interaction,specificity_gap,hotspot_overlap
0,EXAMPLE_DATA_rfdiffusion_0024,rfdiffusion,4.3967,4.0,12.0,0.25
1,EXAMPLE_DATA_rfdiffusion_0008,rfdiffusion,4.1600,9.0,5.0,0.75
2,EXAMPLE_DATA_rfdiffusion_0150,rfdiffusion,4.0967,4.0,16.0,0.25
3,EXAMPLE_DATA_rfdiffusion_0148,rfdiffusion,3.6467,7.0,14.0,0.50
4,EXAMPLE_DATA_rfdiffusion_0193,rfdiffusion,3.2067,7.0,12.0,0.75


## D3 (part 2) checklist
- [ ] Conformational-specificity test run for every design (`results/specificity.csv`): `pae_fibril`, `pae_monomer`, `specificity_gap`.
- [ ] "Fibril-selective" defined (`pae_fibril ≤ cutoff` AND `specificity_gap ≥ GAP_MIN`) and **justified**; selective fraction reported **per paradigm** (figure `results/p11_specificity.png`).
- [ ] Head-to-head: fibril hit rate + interface-energy distribution per paradigm (figure `results/p11_headtohead.png`).
- [ ] Cross-amyloid specificity (tau vs α-syn) computed or scaffolded (`results/cross_amyloid.csv`).
- [ ] `results/top_candidates.csv`: top fibril-selective set per paradigm (or an honest "few/none selective").
- [ ] Honest discussion: most designs fail the monomer counter-test; selectivity is a hypothesis until the assay.

**Next:** `05_validation_plan.ipynb` — the fibril-vs-monomer ELISA/SPR plan + the diagnostic-tracer framing.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation Plan — fibril-vs-monomer ELISA/SPR + controls + diagnostic framing

**Standard slot:** *validation plan.* **For Project 11 this means:** turn the fibril-selective top
candidates into a **costed, controlled wet-lab plan** whose centerpiece is a **fibril-vs-monomer**
selectivity assay (ELISA/SPR), with the mandatory controls (positive conformational antibody,
**scrambled-interface** negative, monomer/unrelated negatives), an expression strategy, and the
**diagnostic-tracer** framing (PET tracer / assay) (D4/D5).

A design that passes every filter — even the in-silico monomer counter-test — is a **hypothesis**. The
**fibril-vs-monomer assay** is what tests it. Needs `results/top_candidates.csv` (notebook 04).

## Setup paths

In [33]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_11_fibril_binder/notebooks


## 1 · Draft the experimental validation plan

Generate a plan card from the top candidates: the fibril-vs-monomer assay, controls, expression,
timeline, costed reagents, and the diagnostic-tracer framing. Fill the `<...>` from your own numbers;
this is the deliverable other people will actually read.

In [34]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_par = top.groupby("paradigm").size().to_dict() if n_top else {}

plan = f"""# Conformation-Specific Fibril-Binder Validation Plan (Project 11 — by <your name>, <date>)

## Candidates
Top {n_top} FIBRIL-SELECTIVE candidates carried forward ({by_par}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured — `pae_interaction` is confidence (not affinity),
and the `specificity_gap` is a model proxy (not a measured fold-selectivity). The monomer is
disordered, so its model is itself uncertain. The fibril-vs-monomer assay below is the real test.

## Target conformations (the whole point)
- ON-target: the AMYLOID FIBRIL (tau PHF from 5O3L/5O3T, or alpha-synuclein fibril from 6CU7/6H6B).
  Prepare recombinant fibrils in vitro (seeded aggregation); confirm fibrils by ThT fluorescence + TEM/cryo-EM.
- OFF-target (counter): the MONOMER of the SAME protein (freshly purified, kept monomeric; verify by SEC).
- OFF-target (cross-amyloid): the OTHER amyloid fibril (tau vs alpha-syn) for diagnostic discrimination.

## Expression strategy
- Binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (50-90 aa) -> high yield expected.
- Antigens: recombinant tau / alpha-synuclein; prepare BOTH a monomer prep AND an in-vitro fibril prep
  from the same construct (this matched pair is what makes the selectivity readout clean).

## Assays (go/no-go -> basic -> the selectivity test)
1. Go/no-go: express binder -> SDS-PAGE -> SEC (monodisperse?).
2. Binding: SPR or BLI vs immobilized FIBRIL -> apparent K_D + kinetics. Test a dilution series.
3. THE SELECTIVITY TEST (the point): fibril-vs-monomer ELISA/SPR -> signal on FIBRIL must be >> signal
   on MONOMER (report the selectivity RATIO measured here; do NOT report the in-silico gap as the result).
4. Cross-amyloid: same readout vs the OTHER amyloid fibril (must be low for a discriminating tracer).
5. (Diagnostic deep dive) tissue staining / fibril pulldown from patient-derived material under approval.

## Controls (MANDATORY)
- Positive: a known conformation-specific anti-fibril antibody/tracer (e.g., a conformational mAb or a
  validated amyloid PET-tracer scaffold) -> assay + fibril prep are active and conformation-discriminating.
- Negative (scrambled-interface): YOUR OWN top design with its fibril-contacting residues scrambled
  -> must LOSE fibril binding (cleanest specificity control).
- Negative (monomer): the MONOMER of the same protein -> a selective binder must NOT bind it.
- Negative (unrelated): an unrelated mini-protein / an unrelated amyloid -> should not bind.

## Diagnostic-tracer framing
The intended use is DIAGNOSTIC (a conformation-selective probe for PET imaging or a fibril-detection
assay) and/or an aggregation MODULATOR — recognizing pathological aggregates, not the physiological
monomer. This is a defensible, in-scope neurodegeneration application (low dual-use). A clinical PET
tracer additionally needs BBB penetration, radiolabeling chemistry, and pharmacokinetics — out of
scope for this capstone but named here as the translational path.

## Realistic expectations
Conformational selectivity is VERY hard: the binder must REJECT the abundant monomer. The MAJORITY of
in-silico "selective" designs will fail the monomer counter-test experimentally. Report the measured
selectivity ratio and the experimental hit rate honestly. Do NOT imply a working tracer or fabricate
a K_D / selectivity number.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} binders + scrambled-interface negatives): $<...>, <...> weeks (IGSC-screened provider).
- Recombinant tau / alpha-syn (monomer + fibril preps) + ThT + TEM time + SPR/BLI chips + positive-control mAb: $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
Conformation-selective binders to pathological amyloid aggregates for neurodegeneration DIAGNOSTICS /
aggregation modulation (in scope; low dual-use). Gene synthesis via a biosecurity-screening provider;
any patient-derived material + wet lab under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:700], "...")

wrote results/validation_plan.md — fill the <...> placeholders from your numbers.
# Conformation-Specific Fibril-Binder Validation Plan (Project 11 — by <your name>, <date>)

## Candidates
Top 5 FIBRIL-SELECTIVE candidates carried forward ({'rfdiffusion': 5}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured — `pae_interaction` is confidence (not affinity),
and the `specificity_gap` is a model proxy (not a measured fold-selectivity). The monomer is
disordered, so its model is itself uncertain. The fibril-vs-monomer assay below is the real test.

## Target conformations (the whole point)
- ON-target: the AMYLOID FIBRIL (tau PHF from 5O3L/5O3T, or alpha-synuclein fibril from 6CU7/6H6B).
  Prepare recombinant fibrils in vitro (seeded aggr ...


## 2 · Build the scrambled-interface negative controls

The single cleanest specificity control: take each top design and **scramble its fibril-contacting
residues** — it should **lose** fibril binding. Generating these alongside the real designs (same
expression batch) makes the fibril-vs-monomer comparison airtight. Here we scaffold the sequence-level
scramble deterministically; on Colab, scramble the *interface* positions specifically using the
predicted fibril contacts.

In [35]:
import random
import binder_tools as bt   # bt._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

def scramble_interface(seq, frac=0.4, seed=0):
    """Deterministically shuffle a fraction of the sequence as a NEGATIVE-CONTROL stand-in.
    On Colab, scramble the predicted FIBRIL-INTERFACE residues specifically (positions contacting the fibril)."""
    rng = random.Random(seed)
    seq = list(seq)
    idx = list(range(len(seq)))
    rng.shuffle(idx)
    k = max(1, int(len(seq) * frac))
    chosen = idx[:k]
    vals = [seq[i] for i in chosen]
    rng.shuffle(vals)
    for i, v in zip(chosen, vals):
        seq[i] = v
    return "".join(seq)

negs = []
if n_top and "sequence" in top.columns:
    for _, r in top.iterrows():
        s = str(r.get("sequence", ""))
        if s and s.lower() != "nan":
            negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                             parent=r["design_id"], paradigm=r.get("paradigm"),
                             sequence=scramble_interface(s, seed=bt._hashints(r["design_id"]) % 10**6),
                             role="scrambled-interface negative control"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-interface negatives")
else:
    print("Run notebook 04 first to produce results/top_candidates.csv with sequences (need fibril-selective hits).")

wrote results/negative_controls.csv: 5 scrambled-interface negatives


## 3 · (Stretch) Boltz-2 affinity on top hits `[stretch]`

Boltz-2 can predict a binding-affinity signal for the top complexes. Use it for **relative ranking +
caveats only** — **never fabricate a K_D**, and never present a predicted number as measured. For a
fibril target this is even less reliable (large, repetitive assembly), so treat it as a tie-breaker for
which selective hits to test first, not as evidence of binding or of selectivity.

In [36]:
# Scaffold ONLY. Do NOT invent affinities or selectivity ratios. On Colab:
#   pip install boltz; build the (binder, fibril) complex input; run boltz predict with affinity mode;
#   read the predicted-affinity signal and report the RELATIVE ranking of the top hits + heavy caveats.
# Pinned upstream (verify): https://github.com/jwohlwend/boltz
print("Boltz-2 affinity is a STRETCH scaffold: relative ranking + caveats only, NEVER a fabricated K_D.")
print("Use it to PRIORITIZE which fibril-selective hits to test first in the assay — not as evidence of binding.")

Boltz-2 affinity is a STRETCH scaffold: relative ranking + caveats only, NEVER a fabricated K_D.
Use it to PRIORITIZE which fibril-selective hits to test first in the assay — not as evidence of binding.


## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: **fibril-vs-monomer** ELISA/SPR (the selectivity test), cross-amyloid readout, expression, timeline, costed reagents.
- [ ] Controls specified: positive (conformational anti-fibril antibody/tracer), **scrambled-interface** negative (`results/negative_controls.csv`), **monomer** negative, unrelated negative.
- [ ] Matched monomer + in-vitro-fibril preps planned (ThT + TEM/cryo-EM confirmation).
- [ ] Diagnostic-tracer framing stated (PET / assay; BBB + radiochemistry named as the translational path).
- [ ] (Stretch) Boltz-2 affinity used only for relative ranking, with caveats — no fabricated K_D / selectivity.
- [ ] Honest framing: every design is a hypothesis until the fibril-vs-monomer assay; report the measured selectivity ratio + experimental hit rate.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a conformation-specific fibril binder set with a fibril-vs-monomer validation plan and a diagnostic-tracer framing.